# 02 — RoBERTa and DistilRoBERTa search and final evaluation

Use **Runtime → Change runtime type → GPU**. This notebook is self-contained and resumes completed runs from Drive. Model weights are downloaded from Hugging Face in Colab.


Use these prepared files from the original project (the outputs of script 09):

```text
MyDrive/thesis_ml/data/dev.csv
MyDrive/thesis_ml/data/supplementare.csv
MyDrive/thesis_ml/data/test.csv        # needed only for final evaluation
```

The first two CSVs need `id`, `thread_id`, `testo`, `etichetta`; `dev.csv` also needs the existing `fold` column numbered 1–5. Labels are `strumentale`, `intrinseco`, `ambiguo`.

If you only have raw annotation files, first obtain the prepared CSVs from the original project; these notebooks do not guess a new split. No original Python scripts are needed once the CSVs are prepared.

Edit the configuration cell and run cells from top to bottom. Results are saved in Drive. Use one session at a time per experiment folder. Final evaluation is initially disabled so **Run all** cannot accidentally use the test set during selection. Enable it only after choosing the complete comparison protocol.

## What is being compared?
Two models × three learning rates × three seeds × five folds = **90 training runs**. Each run records epochs 1–6. Small fine-tuning rates adapt pretrained weights; the grid `1e-5, 2e-5, 5e-5` spans a factor of five. Keep folds fixed while changing training seeds.

The default session completes at most 10 new runs. Rerun the search cell to continue. An interrupted fold restarts from the pretrained model; other completed folds remain saved. This can take several Colab sessions, whose resources vary. You can use `ACTIVE_MODELS` to choose one model per session without changing the experiment grid.

Changed software versions or model revisions create a separate experiment folder to avoid silently mixing incompatible runs. GPU type is recorded for each run; a different GPU can resume but may add numerical variation.

## Install dependencies
Run this cell first in a fresh Colab GPU runtime. Use Colab’s installed PyTorch; do not replace its GPU build. Transformers is pinned to the API used below.

In [ ]:
%pip install -q "transformers==4.57.1" "scikit-learn==1.6.1" "pandas>=2.2,<3" "matplotlib>=3.9,<4"

## Mount Drive and configure the experiment

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = Path('/content/drive/MyDrive/thesis_ml')
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_ROOT = PROJECT_DIR / 'results'
# These form the complete experiment. Changing them creates a new result folder.
MODEL_NAMES = ['roberta-base', 'distilroberta-base']
LEARNING_RATES = [1e-5, 2e-5, 5e-5]
SEEDS = [42, 123, 2026]
MAX_EPOCHS = 6
BATCH_SIZE = 16
MAX_LENGTH = 256
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
USE_MIXED_PRECISION = True
FINAL_SEED = 42
EXPECTED_DEV_ROWS = 641
EXPECTED_TEST_ROWS = 159
RUN_FINAL = False
# Session controls do not change the experiment identity.
# You can run one model per session and keep the same experiment folder.
ACTIVE_MODELS = MODEL_NAMES.copy()  # e.g. ['distilroberta-base']
MAX_RUNS_THIS_SESSION = 10  # 0 = no limit. Rerun later to continue completed-fold progress.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Data, storage and training helpers
Training uses the original weighted loss, batch size and text limit. Mixed precision reduces GPU memory use. The learning-rate schedule always covers six epochs, including when final training stops at an earlier selected epoch.

In [ ]:
from pathlib import Path
import gc, hashlib, importlib.metadata, json, marshal, os, platform, random, re, time, types, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from IPython.display import display

LABELS = ['strumentale', 'intrinseco', 'ambiguo']
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
TEST_NOTE = ('The existing test set was already used in preliminary experiments. '
             'It is not a retrospectively untouched holdout.')

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def safe_name(value):
    # Hugging Face IDs can contain a slash; local checkpoints can have absolute paths.
    return re.sub(r'[^A-Za-z0-9_.-]', '_', str(value)).strip('._')[:80] + '_' + json_hash(str(value))[:8]

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    os.replace(temporary, path)

def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

def read_table(path, with_folds=False, allow_empty=False):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Copy the prepared CSV files from the original project.')
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    needed = {'id', 'thread_id', 'testo', 'etichetta'} | ({'fold'} if with_folds else set())
    if not needed.issubset(frame.columns):
        raise ValueError(f'{path.name}: missing columns {sorted(needed - set(frame.columns))}')
    if frame.empty and not allow_empty:
        raise ValueError(f'{path.name} is empty.')
    if frame['id'].duplicated().any():
        raise ValueError(f'{path.name}: duplicate example IDs.')
    for col in ['id', 'thread_id', 'testo']:
        if frame[col].str.strip().eq('').any():
            raise ValueError(f'{path.name}: blank {col}.')
    if not set(frame['etichetta']).issubset(LABELS):
        raise ValueError(f'{path.name}: unknown labels {set(frame.etichetta) - set(LABELS)}')
    if with_folds:
        numeric = pd.to_numeric(frame['fold'], errors='raise')
        if (numeric % 1 != 0).any():
            raise ValueError('Fold numbers must be integers.')
        frame['fold'] = numeric.astype(int)
        if set(frame.fold) != set(range(1, 6)):
            raise ValueError('Expected the five existing folds numbered 1 to 5.')
        if (frame.groupby('thread_id')['fold'].nunique() > 1).any():
            raise ValueError('A thread appears in more than one fold.')
    return frame

def load_development(data_dir):
    dev = read_table(data_dir / 'dev.csv', with_folds=True)
    supp = read_table(data_dir / 'supplementare.csv', allow_empty=True)
    if EXPECTED_DEV_ROWS is not None and len(dev) != EXPECTED_DEV_ROWS:
        raise ValueError(f'Expected {EXPECTED_DEV_ROWS} development rows, found {len(dev)}.')
    if set(dev.id) & set(supp.id):
        raise ValueError('Development and supplementary example IDs overlap.')
    return dev, supp

def fold_data(dev, supp, fold):
    validation = dev[dev.fold == fold].reset_index(drop=True)
    gold_train = dev[dev.fold != fold].drop(columns='fold')
    eligible = supp[~supp.thread_id.isin(set(validation.thread_id))]
    training = pd.concat([gold_train, eligible], ignore_index=True)
    if set(training.thread_id) & set(validation.thread_id):
        raise ValueError('Training/validation thread overlap.')
    if set(training.etichetta) != set(LABELS):
        raise ValueError('A training fold is missing a class.')
    return training, validation

def metrics_for(truth, predictions):
    report = classification_report(truth, predictions, labels=LABELS,
                                   output_dict=True, zero_division=0)
    return {'macro_f1': float(report['macro avg']['f1-score']),
            'accuracy': float(np.mean(np.asarray(truth) == np.asarray(predictions))),
            'per_class': {label: report[label] for label in LABELS}}

def code_hash(functions):
    # Normalize notebook cell filenames and line positions, preserving executable code.
    def normalize(c):
        constants = tuple(normalize(x) if isinstance(x, types.CodeType) else x for x in c.co_consts)
        line_key = 'co_linetable' if hasattr(c, 'co_linetable') else 'co_lnotab'
        return c.replace(co_filename='', co_firstlineno=1, co_consts=constants, **{line_key: b''})
    return json_hash({f.__name__: hashlib.sha256(marshal.dumps(normalize(f.__code__))).hexdigest()
                      for f in functions})

def experiment_manifest(protocol, functions, extra_versions=None):
    versions = {name: importlib.metadata.version(name)
                for name in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']}
    versions.update(extra_versions or {})
    return {'workflow_version': 1, 'protocol': protocol,
            'data_hashes': {name: file_hash(DATA_DIR / name)
                            for name in ['dev.csv', 'supplementare.csv']},
            'code_hash': code_hash([read_table, load_development, fold_data, metrics_for] + functions),
            'python': platform.python_version(), 'versions': versions}

def start_experiment(manifest, output_root, prefix):
    experiment_id = json_hash(manifest)[:16]
    directory = output_root / f'{prefix}_{experiment_id}'
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / 'manifest.json'
    if path.exists() and json.loads(path.read_text()) != manifest:
        raise ValueError('Manifest mismatch. Use a different experiment folder.')
    atomic_json(path, manifest)
    return directory

def assert_inputs_unchanged():
    for name, expected in MANIFEST['data_hashes'].items():
        if file_hash(DATA_DIR / name) != expected:
            raise ValueError(f'{name} changed during this session. Rerun setup before continuing.')

def read_completed(directory, job, expected_epochs):
    path = directory / 'metrics.json'
    if not path.exists():
        return None
    result = json.loads(path.read_text())
    if result.get('job') != job:
        raise ValueError(f'Job mismatch at {directory}.')
    if result.get('status') != 'complete':
        return None  # Partial runs restart from their original pretrained weights.
    prediction_file = directory / 'predictions.csv'
    if (len(result['epochs']) != expected_epochs or
        not prediction_file.exists() or
        file_hash(prediction_file) != result.get('predictions_hash')):
        raise ValueError(f'Incomplete/corrupt completed result at {directory}.')
    return result

def save_completed(directory, job, epoch_metrics, predictions, elapsed, extra=None):
    directory.mkdir(parents=True, exist_ok=True)
    atomic_csv(directory / 'predictions.csv', predictions)
    value = {'status': 'complete', 'job': job, 'epochs': epoch_metrics,
             'seconds': float(elapsed), 'predictions_hash': file_hash(directory / 'predictions.csv')}
    value.update(extra or {})
    atomic_json(directory / 'metrics.json', value)
    return value

def checked_test(dev, supp):
    # Only call in the final evaluation section, after selection is frozen.
    test = read_table(DATA_DIR / 'test.csv')
    if EXPECTED_TEST_ROWS is not None and len(test) != EXPECTED_TEST_ROWS:
        raise ValueError(f'Expected {EXPECTED_TEST_ROWS} test rows, found {len(test)}.')
    training = pd.concat([dev.drop(columns='fold'), supp], ignore_index=True)
    if set(training.id) & set(test.id) or set(training.thread_id) & set(test.thread_id):
        raise ValueError('Final training/test overlap in IDs or threads.')
    if set(training.etichetta) != set(LABELS):
        raise ValueError('Final training is missing a class.')
    return training, test

def final_identity(selected, test):
    return {'manifest_hash': json_hash(MANIFEST), 'selected': selected,
            'test_hash': file_hash(DATA_DIR / 'test.csv'), 'seed': FINAL_SEED,
            'test_rows': len(test), 'test_note': TEST_NOTE}

def show_confusion(truth, prediction, path):
    matrix = confusion_matrix(truth, prediction, labels=LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(3), LABELS, rotation=20)
    ax.set_yticks(range(3), LABELS)
    ax.set(xlabel='Predicted', ylabel='True')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    display(fig)
    plt.close(fig)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
os.environ.setdefault('USE_TF', '0')
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

class EncodedComments(Dataset):
    def __init__(self, frame, tokenizer):
        self.encoded = tokenizer(frame.testo.tolist(), padding='max_length', truncation=True,
                                 max_length=MAX_LENGTH, return_tensors='pt')
        self.labels = torch.tensor([LABEL2ID[label] for label in frame.etichetta], dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: value[index] for key, value in self.encoded.items()}, 'labels': self.labels[index]}

@torch.no_grad()
def predict_transformer(model, loader, device):
    model.eval()
    predicted = []
    for batch in loader:
        inputs = {key: value.to(device) for key, value in batch.items() if key != 'labels'}
        logits = model(**inputs).logits
        predicted.extend(logits.argmax(dim=1).cpu().tolist())
    return [LABELS[index] for index in predicted]

def train_transformer(model_name, learning_rate, seed, training, tokenizer,
                      epochs_to_train, validation=None, progress_dir=None):
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_ds = EncodedComments(training, tokenizer)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              generator=generator, num_workers=0)
    validation_loader = None
    if validation is not None:
        validation_loader = DataLoader(EncodedComments(validation, tokenizer),
                                       batch_size=BATCH_SIZE * 2, num_workers=0)
    counts = training.etichetta.value_counts()
    if any(counts.get(label, 0) == 0 for label in LABELS):
        raise ValueError('Training is missing a class.')
    weights = torch.tensor([len(training) / (3 * counts[label]) for label in LABELS],
                           dtype=torch.float32, device=device)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, revision=MODEL_REVISIONS.get(model_name), num_labels=3,
        id2label=dict(enumerate(LABELS)), label2id=LABEL2ID).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    # Always retain the six-epoch schedule, including shorter selected final training.
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer,
        num_warmup_steps=int(WARMUP_FRACTION * total_steps), num_training_steps=total_steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    amp_enabled = USE_MIXED_PRECISION and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    history, prediction_frames = [], []
    for epoch in range(1, epochs_to_train + 1):
        started = time.time()
        model.train()
        loss_sum = 0.0
        for batch in train_loader:
            y = batch.pop('labels').to(device)
            inputs = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(**inputs).logits
                loss = loss_fn(logits, y)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite loss at epoch {epoch}. Inspect this run before changing the grid.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()  # Do not advance after a skipped non-finite optimizer step.
            loss_sum += float(loss.detach().cpu())
        record = {'epoch': epoch, 'training_loss': loss_sum / len(train_loader),
                  'seconds': time.time() - started}
        if validation_loader is not None:
            prediction = predict_transformer(model, validation_loader, device)
            record.update(metrics_for(validation.etichetta.tolist(), prediction))
            frame = validation[['id', 'thread_id', 'etichetta']].rename(columns={'etichetta': 'true'}).copy()
            frame['epoch'], frame['predicted'] = epoch, prediction
            prediction_frames.append(frame)
        history.append(record)
        if progress_dir is not None:
            atomic_json(progress_dir / 'progress.json', {'status': 'partial', 'epochs': history,
                        'resume_note': 'If interrupted, this fold restarts; completed folds remain saved.'})
        score_text = f', macro-F1={record["macro_f1"]:.4f}' if 'macro_f1' in record else ''
        print(f'  epoch {epoch}/{epochs_to_train}, loss={record["training_loss"]:.4f}{score_text}, '
              f'{record["seconds"]:.0f}s', flush=True)
    predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else None
    return model, history, predictions

def transformer_job(model_name, learning_rate, seed, fold):
    return {'family': model_name, 'learning_rate': float(learning_rate),
            'seed': int(seed), 'fold': int(fold)}

def transformer_path(model_name, learning_rate, seed, fold):
    return (EXP_DIR / 'runs' / safe_name(model_name) / f'lr_{learning_rate:g}' /
            f'seed_{seed}' / f'fold_{fold}')

def summarize_score_matrix(matrix):
    # Rows are folds and columns are seeds; these are not independent samples.
    return {'mean_macro_f1': float(matrix.mean()),
            'fold_std': float(matrix.mean(axis=1).std(ddof=1)),
            'seed_std': float(matrix.mean(axis=0).std(ddof=1)) if matrix.shape[1] > 1 else None,
            'fold_means': matrix.mean(axis=1).tolist(),
            'seed_means': matrix.mean(axis=0).tolist()}

def transformer_summary():
    rows, selected, run_rows = [], {}, []
    for model_name in MODEL_NAMES:
        model_rows = []
        all_complete = True
        for learning_rate in LEARNING_RATES:
            results = {}
            for fold in range(1, 6):
                for seed in SEEDS:
                    job = transformer_job(model_name, learning_rate, seed, fold)
                    result = read_completed(transformer_path(model_name, learning_rate, seed, fold), job, MAX_EPOCHS)
                    if result is not None:
                        results[(fold, seed)] = result
                        for score in result['epochs']:
                            run_rows.append({**job, 'epoch': score['epoch'], 'macro_f1': score['macro_f1']})
            if len(results) != 5 * len(SEEDS):
                all_complete = False
                continue
            for epoch in range(1, MAX_EPOCHS + 1):
                matrix = np.array([[results[(fold, seed)]['epochs'][epoch - 1]['macro_f1']
                                    for seed in SEEDS] for fold in range(1, 6)])
                row = {'family': model_name, 'learning_rate': float(learning_rate), 'epoch': epoch,
                       **summarize_score_matrix(matrix)}
                rows.append(row)
                model_rows.append(row)
        if all_complete:
            selected[model_name] = sorted(model_rows,
                key=lambda r: (-r['mean_macro_f1'], r['epoch'], r['learning_rate']))[0]
    return pd.DataFrame(rows), selected, pd.DataFrame(run_rows)

## Validate data and initialize experiment

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable Runtime > Change runtime type > GPU, then rerun the notebook.')
if not MODEL_NAMES or len(set(MODEL_NAMES)) != len(MODEL_NAMES):
    raise ValueError('MODEL_NAMES must contain unique model identifiers.')
if not LEARNING_RATES or len(set(LEARNING_RATES)) != len(LEARNING_RATES) or any(lr <= 0 for lr in LEARNING_RATES):
    raise ValueError('Learning rates must be distinct positive numbers.')
if not SEEDS or len(set(SEEDS)) != len(SEEDS) or any(not isinstance(s, int) or s < 0 or s >= 2**32 for s in SEEDS):
    raise ValueError('Use distinct integer seeds between 0 and 2**32-1.')
if not set(ACTIVE_MODELS).issubset(MODEL_NAMES) or MAX_RUNS_THIS_SESSION < 0:
    raise ValueError('Invalid session controls.')
if MAX_EPOCHS < 1 or BATCH_SIZE < 1 or MAX_LENGTH < 1 or not 0 <= WARMUP_FRACTION < 1:
    raise ValueError('Invalid training settings.')
dev, supp = load_development(DATA_DIR)
DEVICE_NAME = torch.cuda.get_device_name(0)
MODEL_REVISIONS = {name: getattr(AutoConfig.from_pretrained(name), '_commit_hash', None)
                   for name in MODEL_NAMES}
MANIFEST = experiment_manifest(
    {'kind': 'transformers', 'models': MODEL_NAMES, 'learning_rates': LEARNING_RATES,
     'seeds': SEEDS, 'folds': [1, 2, 3, 4, 5], 'labels': LABELS,
     'max_epochs': MAX_EPOCHS, 'batch_size': BATCH_SIZE, 'max_length': MAX_LENGTH,
     'weight_decay': WEIGHT_DECAY, 'warmup_fraction': WARMUP_FRACTION,
     'mixed_precision': USE_MIXED_PRECISION, 'weighted_loss': True,
     'final_seed': FINAL_SEED, 'model_revisions': MODEL_REVISIONS,
     'selection': 'mean_macro_f1_by_common_epoch; ties: fewer epochs then smaller learning rate'},
    [seed_everything, EncodedComments.__init__, EncodedComments.__getitem__,
     predict_transformer.__wrapped__, train_transformer, transformer_job,
     summarize_score_matrix, transformer_summary],
    {'torch': torch.__version__, 'transformers': importlib.metadata.version('transformers'),
     'cuda': str(torch.version.cuda)})
EXP_DIR = start_experiment(MANIFEST, OUTPUT_ROOT, 'transformers')
print('GPU:', DEVICE_NAME)
print('Results:', EXP_DIR)
print(f'Full experiment: {len(MODEL_NAMES) * len(LEARNING_RATES) * len(SEEDS) * 5} '
      f'training runs, each {MAX_EPOCHS} epochs. test.csv is not loaded.')
print('Completed runs resume under the same data, configuration, model revisions, code and versions.')
print('GPU type is recorded per run; changing it can introduce additional numerical variation.')

## Run or resume a search session
Fresh pretrained weights are loaded for every fold/seed/configuration. Results and partial progress are written to Drive. No test labels are read.

In [ ]:
def run_search_session():
    assert_inputs_unchanged()
    completed_this_session = 0
    for model_name in ACTIVE_MODELS:
        pending = [(lr, seed, fold) for lr in LEARNING_RATES for seed in SEEDS for fold in range(1, 6)
                   if read_completed(transformer_path(model_name, lr, seed, fold),
                                     transformer_job(model_name, lr, seed, fold), MAX_EPOCHS) is None]
        if not pending:
            print(model_name, ': all runs complete.')
            continue
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=MODEL_REVISIONS.get(model_name))
        for lr, seed, fold in pending:
            if MAX_RUNS_THIS_SESSION and completed_this_session >= MAX_RUNS_THIS_SESSION:
                print('Session limit reached. Rerun this cell or reconnect later to continue.')
                return
            directory = transformer_path(model_name, lr, seed, fold)
            training, validation = fold_data(dev, supp, fold)
            print(f'\n{model_name}: lr={lr:g}, seed={seed}, fold={fold}', flush=True)
            started = time.time()
            model, history, predictions = train_transformer(model_name, lr, seed, training,
                tokenizer, MAX_EPOCHS, validation=validation, progress_dir=directory)
            save_completed(directory, transformer_job(model_name, lr, seed, fold),
                           history, predictions, time.time() - started,
                           {'training_rows': len(training), 'validation_rows': len(validation),
                            'model_revision': getattr(model.config, '_commit_hash', None),
                            'device': DEVICE_NAME})
            del model
            gc.collect()
            torch.cuda.empty_cache()
            completed_this_session += 1
    print('Requested session work complete.')

run_search_session()

## Summarize and select model settings
For each model, compare every learning-rate/epoch pair using mean macro-F1 across all five folds and all seeds. Choose **one common epoch**, rather than separately taking each fold’s maximum score. Exact ties prefer fewer epochs, then the smaller rate. A model is selectable only after its full grid completes.

Fold spread and seed spread are reported separately. These scores are development results used for selection; they are not unbiased final estimates or independent statistical samples.

In [ ]:
assert_inputs_unchanged()
summary, selected, runs = transformer_summary()
atomic_csv(EXP_DIR / 'run_metrics.csv', runs)
atomic_csv(EXP_DIR / 'candidate_summary.csv', summary)
document = {'manifest_hash': json_hash(MANIFEST), 'data_hashes': MANIFEST['data_hashes'],
            'complete': len(selected) == len(MODEL_NAMES), 'selected': selected,
            'expected_models': MODEL_NAMES,
            'note': 'Epoch and hyperparameters were selected on CV; these are development scores.'}
atomic_json(EXP_DIR / 'selected_configs.json', document)
if not summary.empty:
    display(summary.drop(columns=['fold_means', 'seed_means']).sort_values(
        ['family', 'mean_macro_f1'], ascending=[True, False]))
    for model_name in MODEL_NAMES:
        subset = summary[summary.family == model_name]
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        for lr in LEARNING_RATES:
            curve = subset[subset.learning_rate == lr].sort_values('epoch')
            if not curve.empty:
                ax.plot(curve.epoch, curve.mean_macro_f1, marker='o', label=f'lr={lr:g}')
        ax.set(title=model_name, xlabel='Epoch', ylabel='Mean validation macro-F1 across folds and seeds')
        ax.legend()
        fig.tight_layout()
        fig.savefig(EXP_DIR / f'{safe_name(model_name)}_learning_curves.png', dpi=180)
        display(fig)
        plt.close(fig)
print('Completed model selections:', json.dumps(selected, indent=2))
if not document['complete']:
    print('Experiment incomplete. Resume the search; final evaluation remains disabled until all models finish.')

## Compare against the baseline search
Notebook 01 must use the same prepared data. This cell combines development results when exactly one matching baseline experiment is available.

In [ ]:
# Optional development comparison with notebook 01. Require matching input data.
baseline_selections = sorted(OUTPUT_ROOT.glob('baselines_*/selected_configs.json'))
matching_baselines = []
for path in baseline_selections:
    baseline_document = json.loads(path.read_text())
    if baseline_document.get('data_hashes') == MANIFEST['data_hashes']:
        matching_baselines.append(path)
if len(matching_baselines) == 1 and len(selected) == len(MODEL_NAMES):
    baseline_document = json.loads(matching_baselines[0].read_text())
    comparison = pd.DataFrame(list(baseline_document['selected'].values()) + list(selected.values()))
    comparison = comparison.sort_values('mean_macro_f1', ascending=False)
    atomic_csv(EXP_DIR / 'development_model_comparison.csv', comparison)
    display(comparison.drop(columns=['fold_scores', 'fold_means', 'seed_means'], errors='ignore'))
elif len(matching_baselines) > 1:
    print('Multiple matching baseline experiments. Choose the intended selected_configs.json explicitly.')
else:
    print('Run notebook 01 and complete both transformer searches for the combined development table.')

## Final training and test evaluation — optional
After both searches complete and the comparison protocol is frozen, set `RUN_FINAL=True`. Each selected transformer is retrained from fresh pretrained weights on all development plus supplementary examples, using the predeclared final seed 42. The same separate test evaluates both. Keep the original disclosure that this test had already been used in preliminary experiments. Test scores must not be used for further selection.

In [ ]:
if not RUN_FINAL:
    print('Final training skipped. Set RUN_FINAL=True only after all searches and the comparison protocol are frozen.')
else:
    assert_inputs_unchanged()
    saved = json.loads((EXP_DIR / 'selected_configs.json').read_text())
    if saved['manifest_hash'] != json_hash(MANIFEST) or not saved['complete']:
        raise ValueError('A complete selection from this experiment is required.')
    training, test = checked_test(dev, supp)
    for model_name in MODEL_NAMES:
        configuration = saved['selected'][model_name]
        directory = EXP_DIR / 'final' / safe_name(model_name)
        identity = final_identity(configuration, test)
        result_path = directory / 'evaluation.json'
        if result_path.exists():
            previous = json.loads(result_path.read_text())
            if previous['identity'] != identity:
                raise ValueError('Existing final evaluation belongs to different settings or test data.')
            print('Final evaluation already complete:', result_path)
            display(pd.DataFrame(previous['report']).T)
            continue
        directory.mkdir(parents=True, exist_ok=True)
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=MODEL_REVISIONS.get(model_name))
        model, history, _ = train_transformer(model_name, configuration['learning_rate'],
            FINAL_SEED, training, tokenizer, configuration['epoch'], progress_dir=directory)
        checkpoint = directory / 'checkpoint'
        model.save_pretrained(checkpoint)
        tokenizer.save_pretrained(checkpoint)
        test_loader = DataLoader(EncodedComments(test, tokenizer), batch_size=BATCH_SIZE * 2)
        prediction = predict_transformer(model, test_loader, next(model.parameters()).device)
        report = classification_report(test.etichetta, prediction, labels=LABELS,
                                       output_dict=True, zero_division=0)
        predictions = test[['id', 'thread_id', 'etichetta']].rename(columns={'etichetta': 'true'}).copy()
        predictions['predicted'] = prediction
        atomic_csv(directory / 'test_predictions.csv', predictions)
        show_confusion(test.etichetta, prediction, directory / 'confusion_matrix.png')
        atomic_json(result_path, {'identity': identity, 'report': report, 'training_history': history,
                    'training_rows': len(training), 'test_rows': len(test), 'checkpoint': str(checkpoint),
                    'device': DEVICE_NAME})
        display(pd.DataFrame(report).T)
        print('Checkpoint:', checkpoint)
        del model
        gc.collect()
        torch.cuda.empty_cache()

## Use the final checkpoint
Each model’s final directory contains `checkpoint/`, which can be loaded with `AutoTokenizer.from_pretrained(path)` and `AutoModelForSequenceClassification.from_pretrained(path)`. Use that explicit checkpoint path in subsequent corpus inference. No model is automatically uploaded to the Hugging Face Hub.

Sources: [RoBERTa paper](https://arxiv.org/abs/1907.11692), [training-seed study](https://arxiv.org/abs/2002.06305), [Colab resource FAQ](https://research.google.com/colaboratory/faq.html).